In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','qa','prod'],label='Select Environment')
env=dbutils.widgets.get('environment')
print(f'Environment is {env}')


In [0]:
bronzetable=f'saleslake_{env}.bronze_{env}.rawproduct'
print(bronzetable)
silverTable=f'saleslake_{env}.silver_{env}.cleanedproduct'
print(silverTable)

In [0]:
from pyspark.sql import functions as F

bronze_df=spark.read.table(bronzetable)
# display(bronze_df)
string_cols=["sku", "product_name", "category", "sub_category","brand", "supplier","status"]
dec_cols=["unit_cost", "list_price"]
date_cols=["launch_date"]
int_cols=["product_id"]
for c in string_cols:
  bronze_df=bronze_df.withColumn(c,F.upper(F.trim(F.col(c).cast("string"))))
for c in dec_cols:
    bronze_df=bronze_df.withColumn(c,F.trim(F.col(c)).cast("decimal(18,2)"))
for c in date_cols:
    bronze_df=bronze_df.withColumn(c,F.to_date(F.trim(F.col(c)),"yyyy-MM-dd"))
for c in int_cols:
    bronze_df=bronze_df.withColumn(c,F.col(c).cast("int"))
bronze_df=bronze_df.withColumn("ingest_ts",F.current_timestamp())
max_ingest_ts = spark.table(silverTable).agg(
    F.coalesce(F.max("ingest_ts"), F.to_timestamp(F.lit("1990-01-01"), "yyyy-MM-dd"))
).collect()[0][0]

df_filtered = bronze_df.filter(F.col("ingest_ts") > max_ingest_ts)

(df_filtered.write
   .format("delta")
   .mode("append")                   # append new incremental rows
   .option("mergeSchema", "true")    # allow schema evolution
   .saveAsTable(silverTable))




In [0]:
%sql
select * from saleslake_dev.silver_dev.cleanedproduct